In [4]:
import pandas as pd

df_baseline = pd.read_csv('data/taiwan_baseline.csv')
print(df_baseline[['hts_code', 'chip_category', 'taiwan_import_share_pct']])

   hts_code                    chip_category  taiwan_import_share_pct
0      8541          Discrete Semiconductors                      3.9
1      8542              Integrated Circuits                     23.3
2      8543  Specialized Electrical Machines                      4.0


In [5]:
# Define scenarios
scenarios = {
    '30-day': {'disruption_rate': 0.40, 'inventory_buffer_factor': 0.50},
    '60-day': {'disruption_rate': 0.65, 'inventory_buffer_factor': 0.25},
    '90-day': {'disruption_rate': 0.85, 'inventory_buffer_factor': 0.00},
}

# Build scenario output
rows = []
for scenario, params in scenarios.items():
    for _, row in df_baseline.iterrows():
        shock = (row['taiwan_import_share_pct'] / 100) * params['disruption_rate'] * (1 - params['inventory_buffer_factor'])
        days_to_critical = 30 if scenario == '90-day' else (60 if scenario == '60-day' else 90)
        rows.append({
            'scenario': scenario,
            'hts_code': row['hts_code'],
            'chip_category': row['chip_category'],
            'taiwan_import_share_pct': row['taiwan_import_share_pct'],
            'disruption_rate': params['disruption_rate'],
            'inventory_buffer_factor': params['inventory_buffer_factor'],
            'supply_shock_pct': round(shock * 100, 2),
            'days_to_critical_shortage': days_to_critical
        })

df_scenarios = pd.DataFrame(rows)
print(df_scenarios[['scenario', 'chip_category', 'supply_shock_pct', 'days_to_critical_shortage']])

  scenario                    chip_category  supply_shock_pct  \
0   30-day          Discrete Semiconductors              0.78   
1   30-day              Integrated Circuits              4.66   
2   30-day  Specialized Electrical Machines              0.80   
3   60-day          Discrete Semiconductors              1.90   
4   60-day              Integrated Circuits             11.36   
5   60-day  Specialized Electrical Machines              1.95   
6   90-day          Discrete Semiconductors              3.31   
7   90-day              Integrated Circuits             19.80   
8   90-day  Specialized Electrical Machines              3.40   

   days_to_critical_shortage  
0                         90  
1                         90  
2                         90  
3                         60  
4                         60  
5                         60  
6                         30  
7                         30  
8                         30  


In [6]:
df_scenarios.to_csv('data/scenario_output.csv', index=False)
print("Saved scenario_output.csv")
print("\nKey finding: 8542 Integrated Circuits supply shock by scenario:")
print(df_scenarios[df_scenarios['hts_code']==8542][['scenario','supply_shock_pct','days_to_critical_shortage']])

Saved scenario_output.csv

Key finding: 8542 Integrated Circuits supply shock by scenario:
  scenario  supply_shock_pct  days_to_critical_shortage
1   30-day              4.66                         90
4   60-day             11.36                         60
7   90-day             19.80                         30


In [10]:
df_dod = pd.read_csv('data/dod_contracts_raw.csv', low_memory=False)

print(df_dod[['recipient_name', 'naics_code', 'naics_description', 
              'total_dollars_obligated', 'product_or_service_code']].head(10))

                                     recipient_name  naics_code  \
0                                    SAVE AGAIN.COM      334111   
1                      SOUTHWEST RESEARCH INSTITUTE      334519   
2  BAE SYSTEMS TECHNOLOGY SOLUTIONS & SERVICES INC.      334511   
3                    STERLING COMPUTERS CORPORATION      334111   
4                                STELLANT PST CORP.      334515   
5                           ANIXTER MIDDLE EAST FZE      334210   
6                                   HPI FEDERAL LLC      334111   
7                   KATMAI DIVERSIFIED SERVICES LLC      334111   
8                                 WIRELESS CCTV LLC      334511   
9  BAE SYSTEMS TECHNOLOGY SOLUTIONS & SERVICES INC.      334511   

                                   naics_description  total_dollars_obligated  \
0                  ELECTRONIC COMPUTER MANUFACTURING                174808.67   
1  OTHER MEASURING AND CONTROLLING DEVICE MANUFAC...               2928656.87   
2  SEARCH, DETECTIO

In [11]:
# Group by recipient and NAICS, sum contract values
df_grouped = df_dod.groupby(['recipient_name', 'naics_code', 'naics_description'])['total_dollars_obligated'].sum().reset_index()
df_grouped.columns = ['recipient_name', 'naics_code', 'naics_description', 'total_contract_value']

# Sort by total value, show top 15
df_grouped = df_grouped.sort_values('total_contract_value', ascending=False)
print(df_grouped.head(15))

                                         recipient_name  naics_code  \
2277                                   RAYTHEON COMPANY      334511   
1646                        LOCKHEED MARTIN CORPORATION      334511   
1972               NORTHROP GRUMMAN SYSTEMS CORPORATION      334511   
2269                                   RAYTHEON COMPANY      334220   
1966               NORTHROP GRUMMAN SYSTEMS CORPORATION      334220   
336   BAE SYSTEMS INFORMATION AND ELECTRONIC SYSTEMS...      334511   
2736                        TEXTRON SYSTEMS CORPORATION      334511   
1962                      NORTHROP GRUMMAN SYSTEMS CORP      334220   
1971               NORTHROP GRUMMAN SYSTEMS CORPORATION      334419   
2270                                   RAYTHEON COMPANY      334290   
1648                        LOCKHEED MARTIN CORPORATION      334519   
1588                    LAUREL TECHNOLOGIES PARTNERSHIP      334511   
88                 ADVANCED ELECTRONICS COMPANY LIMITED      334220   
933   

In [12]:
# Taiwan dependency by NAICS code
# Based on chip intensity of each category
naics_taiwan_dependency = {
    334511: 0.233,  # Navigation/guidance — heavily dependent on 8542 integrated circuits
    334220: 0.100,  # Radio/wireless — moderate dependency
    334419: 0.150,  # Other electronic components — moderate
    334290: 0.080,  # Other comms equipment — lower
    334111: 0.233,  # Electronic computers — high, same as 8542
    334519: 0.080,  # Other measuring devices — lower
    334515: 0.150,  # Instruments — moderate
    334413: 0.233,  # Semiconductors — direct dependency
    334412: 0.233,  # Bare circuit boards — high
    334210: 0.080,  # Telephone apparatus — lower
}

# Map dependency to grouped data
df_grouped['taiwan_dependency_pct'] = df_grouped['naics_code'].map(naics_taiwan_dependency).fillna(0.10)

# Calculate exposure score
df_grouped['exposure_score'] = df_grouped['total_contract_value'] * df_grouped['taiwan_dependency_pct']

# Get top 10 by exposure score
df_defense = df_grouped.sort_values('exposure_score', ascending=False).head(10).reset_index(drop=True)
print(df_defense[['recipient_name', 'naics_code', 'total_contract_value', 'taiwan_dependency_pct', 'exposure_score']])

                                      recipient_name  naics_code  \
0                                   RAYTHEON COMPANY      334511   
1                        LOCKHEED MARTIN CORPORATION      334511   
2               NORTHROP GRUMMAN SYSTEMS CORPORATION      334511   
3  BAE SYSTEMS INFORMATION AND ELECTRONIC SYSTEMS...      334511   
4                        TEXTRON SYSTEMS CORPORATION      334511   
5                                   RAYTHEON COMPANY      334220   
6               NORTHROP GRUMMAN SYSTEMS CORPORATION      334220   
7               NORTHROP GRUMMAN SYSTEMS CORPORATION      334419   
8                      NORTHROP GRUMMAN SYSTEMS CORP      334220   
9                    LAUREL TECHNOLOGIES PARTNERSHIP      334511   

   total_contract_value  taiwan_dependency_pct  exposure_score  
0          4.340387e+10                  0.233    1.011310e+10  
1          2.975899e+10                  0.233    6.933846e+09  
2          1.517938e+10                  0.233    3.5367

In [13]:
df_defense_save = df_defense[['recipient_name', 'naics_code', 'naics_description', 
                               'total_contract_value', 'taiwan_dependency_pct', 'exposure_score']].copy()
df_defense_save.columns = ['program_name', 'naics_code', 'naics_description',
                            'contract_value_usd', 'taiwan_dependency_pct', 'exposure_score']
df_defense_save.to_csv('data/defense_exposure.csv', index=False)
print("Saved defense_exposure.csv")

Saved defense_exposure.csv


In [14]:
# Medical device exposure scoring
medical_data = {
    'device_category': ['Implantables', 'Imaging', 'Monitoring', 'Ventilators', 'Diagnostics'],
    'dependency_score': [5, 5, 4, 4, 3],
    'taiwan_dependency_pct': [0.233, 0.233, 0.150, 0.150, 0.100],
}

df_medical = pd.DataFrame(medical_data)

# Calculate shortage onset days under 90-day blockade
# Higher dependency = faster shortage onset
df_medical['supply_shock_pct'] = (df_medical['taiwan_dependency_pct'] * 0.85 * 1.0 * 100).round(1)
df_medical['shortage_onset_days'] = (90 * (1 - df_medical['taiwan_dependency_pct'])).round(0).astype(int)
df_medical['patient_impact_severity'] = ['Critical', 'High', 'High', 'Critical', 'Moderate']

print(df_medical)

  device_category  dependency_score  taiwan_dependency_pct  supply_shock_pct  \
0    Implantables                 5                  0.233              19.8   
1         Imaging                 5                  0.233              19.8   
2      Monitoring                 4                  0.150              12.8   
3     Ventilators                 4                  0.150              12.8   
4     Diagnostics                 3                  0.100               8.5   

   shortage_onset_days patient_impact_severity  
0                   69                Critical  
1                   69                    High  
2                   76                    High  
3                   76                Critical  
4                   81                Moderate  


In [15]:
df_medical.to_csv('data/medical_exposure.csv', index=False)
print("Saved medical_exposure.csv")
print("\nMedical device shortage timeline at 90-day blockade:")
print(df_medical[['device_category', 'shortage_onset_days', 'patient_impact_severity']])

Saved medical_exposure.csv

Medical device shortage timeline at 90-day blockade:
  device_category  shortage_onset_days patient_impact_severity
0    Implantables                   69                Critical
1         Imaging                   69                    High
2      Monitoring                   76                    High
3     Ventilators                   76                Critical
4     Diagnostics                   81                Moderate
